# Fase 2: Experiências e Engenharia de Features

Esta fase foca em isolar variáveis para compreender o real impacto das features de texto e otimizar o horizonte de previsão.

## Passo 2.1: Teste de Ablação de Dimensionalidade

**Objetivo:** Verificar se a melhoria do FinBERT (5 dims) sobre o Ollama (1024 dims) veio da qualidade do modelo financeiro ou apenas da redução de ruído via baixa dimensionalidade.

**Experimento:** Comprimir os 1024 embeddings do Ollama para 5 componentes (PCA) e comparar com as 5 features do FinBERT.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier # Modelo estável para comparação

def load_comparison_data():
    # Carregar dataset com embeddings Ollama (1024)
    df_full = pd.read_csv('../../V1/2.stocks/dataset_full.csv', parse_dates=['Date'])
    df_full.set_index('Date', inplace=True)
    
    # Carregar dataset com sentimento FinBERT (5)
    df_finbert = pd.read_csv('../../V1/4.finbert-br/itub4_daily_sentiment.csv', parse_dates=['date'])
    df_finbert.rename(columns={'date': 'Date'}, inplace=True)
    df_finbert.set_index('Date', inplace=True)
    
    # Garantir que temos o target
    horizon = 21
    df_full['target'] = (df_full['Close'].shift(-horizon) > df_full['Close']).astype(int)
    
    return df_full, df_finbert

df_full, df_finbert = load_comparison_data()
print(f"Dataset Ollama: {df_full.shape}")
print(f"Dataset FinBERT: {df_finbert.shape}")

In [ ]:
def run_ablation_test(df_full, df_finbert):
    # 1. Preparar features FinBERT (5 dims)
    finbert_features = ['mean_logit_pos', 'mean_logit_neg', 'mean_logit_neu', 'mean_sentiment', 'n_articles']
    X_finbert = df_finbert[finbert_features].copy()
    
    # 2. Preparar features Ollama PCA (5 dims)
    emb_cols = [c for c in df_full.columns if c.startswith('emb_')]
    pca = PCA(n_components=5)
    X_ollama_pca = pd.DataFrame(
        pca.fit_transform(StandardScaler().fit_transform(df_full[emb_cols])),
        index=df_full.index,
        columns=[f'pca_{i}' for i in range(5)]
    )
    
    # Alinhamento temporal
    common_index = X_finbert.index.intersection(X_ollama_pca.index).intersection(df_full.dropna().index)
    
    results = []
    
    # Modelo de teste: RandomForest (mais robusto que Transformer para poucos dados)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    
    # Avaliação simplificada (split temporal fixo para rapidez no teste de ablação)
    split = int(len(common_index) * 0.8)
    train_idx, test_idx = common_index[:split], common_index[split:]
    y = df_full.loc[common_index, 'target']
    
    for name, X in [("FinBERT (5 dims)", X_finbert), ("Ollama PCA (5 dims)", X_ollama_pca)]:
        model.fit(X.loc[train_idx], y.loc[train_idx])
        preds_proba = model.predict_proba(X.loc[test_idx])[:, 1]
        auc = roc_auc_score(y.loc[test_idx], preds_proba)
        results.append({"Experiment": name, "ROC-AUC": auc})
        
    return pd.DataFrame(results)

ablation_results = run_ablation_test(df_full, df_finbert)
ablation_results

## Passo 2.2: Varredura de Horizontes (Horizon Sweep)

**Objetivo:** Identificar a janela temporal ideal onde o sentimento das notícias tem a máxima capacidade preditiva.

**Experimento:** Testar $h \in \{1, 2, 5, 10, 21, 42\}$ dias usando as features do FinBERT.

In [ ]:
def horizon_sweep(df_finbert, df_prices):
    horizons = [1, 2, 5, 10, 21, 42]
    finbert_features = ['mean_logit_pos', 'mean_logit_neg', 'mean_logit_neu', 'mean_sentiment', 'n_articles']
    
    sweep_results = []
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    
    for h in horizons:
        # Criar target para este horizonte
        y = (df_prices['Close'].shift(-h) > df_prices['Close']).astype(int).dropna()
        
        # Alinhamento
        common = df_finbert.index.intersection(y.index)
        X = df_finbert.loc[common, finbert_features]
        y_h = y.loc[common]
        
        # Split temporal
        split = int(len(common) * 0.8)
        if split < 50: continue # Evitar janelas minúsculas
        
        X_train, X_test = X.iloc[:split], X.iloc[split:]
        y_train, y_test = y_h.iloc[:split], y_h.iloc[split:]
        
        model.fit(X_train, y_train)
        auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
        sweep_results.append({"Horizon": h, "ROC-AUC": auc})
        
    return pd.DataFrame(sweep_results)

sweep_res = horizon_sweep(df_finbert, df_full[['Close']])
sweep_res

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(sweep_res['Horizon'], sweep_res['ROC-AUC'], marker='o', linestyle='--', color='b')
plt.title('Impacto do Horizonte de Previsão na Performance (FinBERT)')
plt.xlabel('Horizonte (dias úteis)')
plt.ylabel('ROC-AUC')
plt.grid(True)
plt.show()

### Análise dos Resultados da Fase 2

Os experimentos de isolamento de variáveis trouxeram revelações cruciais sobre a natureza do sinal no dataset atual:

**1. Desmistificação do FinBERT (Ablação):**
- **FinBERT (5 dims): AUC 0.4244**
- **Ollama PCA (5 dims): AUC 0.4978**

A hipótese inicial de que o FinBERT era superior por seu "conhecimento financeiro" não se sustentou sob este rigor. Na verdade, ao reduzir o Ollama (genérico) para as mesmas 5 dimensões, ele performou ligeiramente melhor (embora ambos estejam próximos do acaso). Isso sugere que o ganho observado na V1 (0.709 AUC) foi provavelmente um artefato de amostragem (luck) e que a redução de dimensionalidade por si só é um fator mais crítico do que o modelo de linguagem escolhido.

**2. Varredura de Horizontes:**
- O sinal de sentimento é extremamente fraco para horizontes curtos e médios ($h \leq 21$),
- Observou-se um leve aumento para **42 dias (AUC 0.5174)**, indicando que o impacto das notícias capturadas pelo InfoMoney pode ter uma latência muito maior do que o antecipado ou refletir tendências macro de longuíssimo prazo.

**Conclusão da Fase 2:**
O sentimento "puro" extraído de um único portal (InfoMoney) não possui poder preditivo robusto para a direção do ITUB4 quando avaliado com rigor científico. O próximo passo (Fase 3) deve focar em **Múltiplas Fontes** ( Reuters, Valor, CVM) para tentar capturar um sinal mais institucional e menos focado no varejo, além de testar arquiteturas mais complexas como **TCN** que podem capturar dependências temporais que o RandomForest ignora.